# 23. HR 퇴사 예측 배깅 앙상블 (Bagging HR)

`hr_data.csv`의 퇴사 여부(`attrition`)를 타겟으로 분리(누수 방지)한 뒤 KNN, DecisionTree, SVM 기반 `BaggingClassifier`의 분류 성능을 비교합니다.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

candidates = [
    Path('Study_Python/[7] Machine_Learning/data/hr_data.csv'),
    Path('../[7] Machine_Learning/data/hr_data.csv'),
    Path('data/hr_data.csv'),
]
csv_path = next((p for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError('hr_data.csv 파일을 찾을 수 없습니다.')

data = pd.read_csv(csv_path)
y = data['attrition'].map({'No': 0, 'Yes': 1})
# 타겟(attrition)을 반드시 제외해 데이터 누수 방지
numeric_features = [c for c in data.select_dtypes(include=['int64', 'float64']).columns if c != 'attrition']
X = data[numeric_features].dropna(axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print('학습 데이터 크기:', X_train.shape, '테스트 데이터 크기:', X_test.shape)

In [ ]:
# 1. KNN 기반 배깅 (거리 기반 모델이므로 파이프라인 내 표준화)
knn_bagging = BaggingClassifier(
    estimator=Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=7))]),
    n_estimators=10,
    random_state=42,
)
knn_bagging.fit(X_train, y_train)
print(f'Bagging + KNN Accuracy: {accuracy_score(y_test, knn_bagging.predict(X_test)):.4f}')

# 2. Decision Tree 기반 배깅
dt_bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=10,
    random_state=42,
)
dt_bagging.fit(X_train, y_train)
print(f'Bagging + DT Accuracy:  {accuracy_score(y_test, dt_bagging.predict(X_test)):.4f}')

# 3. SVM 기반 배깅
svm_bagging = BaggingClassifier(
    estimator=Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
    n_estimators=10,
    random_state=42,
)
svm_bagging.fit(X_train, y_train)
print(f'Bagging + SVM Accuracy: {accuracy_score(y_test, svm_bagging.predict(X_test)):.4f}')